# DCOPF Example 5 - Economic Dispatch with Ramp Constraints
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e5):** Pure economic dispatch with **ramp-rate constraints**. No network equations. Initial dispatch: G1=30 MW, G3=70 MW. Load = 107 MW. Ramp rate = 1 MW/min, dispatch interval = 5 min → max ramp = ±5 MW.

In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
deltaT = 5  # dispatch interval (minutes)

# Total system load (MW)
total_load = 107

# Generators: {index: {bus, Pgmin, Pgmax, cost, ramp MW/min, Pg_init MW}}
gen_data = {
    1: {'bus': 1, 'Pgmin': 30, 'Pgmax': 80, 'c': 50, 'ramp': 1, 'Pg_init': 30},
    2: {'bus': 3, 'Pgmin': 20, 'Pgmax': 90, 'c': 10, 'ramp': 1, 'Pg_init': 70},
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

model.GEN = Set(initialize=gen_data.keys())

model.gen_min    = Param(model.GEN, initialize={g: gen_data[g]['Pgmin']   for g in gen_data})
model.gen_max    = Param(model.GEN, initialize={g: gen_data[g]['Pgmax']   for g in gen_data})
model.gen_OpCost = Param(model.GEN, initialize={g: gen_data[g]['c']       for g in gen_data})
model.gen_ramp   = Param(model.GEN, initialize={g: gen_data[g]['ramp']    for g in gen_data})
model.gen_Pinit  = Param(model.GEN, initialize={g: gen_data[g]['Pg_init'] for g in gen_data})

model.G = Var(model.GEN)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
# System-wide power balance
model.SysWidePowerBalance = Constraint(
    expr=sum(model.G[g] for g in model.GEN) == total_load
)

# Generator capacity limits
def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

# Ramp-rate limits: |G[g] - Pg_init[g]| <= ramp[g] * deltaT
def genRampLimits_rule(model, g):
    return inequality(
        -model.gen_ramp[g] * deltaT,
        model.G[g] - model.gen_Pinit[g],
        model.gen_ramp[g] * deltaT
    )
model.genRampLimits = Constraint(model.GEN, rule=genRampLimits_rule)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\n=== Generator Dispatch ===')
for g in model.GEN:
    ramp_used = model.G[g].value - gen_data[g]['Pg_init']
    max_ramp  = gen_data[g]['ramp'] * deltaT
    print(f'  G[{g}] = {model.G[g].value:.4f} MW  '
          f'(init={gen_data[g]["Pg_init"]} MW, ramp={ramp_used:+.4f} MW, limit=\u00b1{max_ramp} MW)')

total_cost = sum(gen_data[g]['c'] * model.G[g].value for g in model.GEN)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')